In [1]:
import geopandas as gpd
import xarray as xr
from rasterstats import zonal_stats
import pandas as pd
import numpy as np
import rioxarray

import matplotlib.pyplot as plt

import glob
import os
from datetime import datetime

In [2]:
#change the working directory to the input folder where the netcdf files are stored
os.chdir("../")
print(os.getcwd())

c:\Users\uginet\Documents\anemie\code_archive\replication_code_JEEM_anemia-main\0_input


In [3]:
ref_date = pd.Timestamp("1900-01-01")

confinement_date = pd.Timestamp("2020-03-25")

confinement_date-ref_date

Timedelta('43913 days 00:00:00')

## The following code extracts temperature data at the location of every DHS cluster, and compute the number of hours per day spent in each of the following bin : 
## (-inf,-30), [T, T+1) \forall T\in[-30,45], [45,+inf)

## First, handle air temperature

In [ ]:
# 1. Load the list of DHs cluster points
dhs_cells = pd.read_csv('./matching_grid_cells_DHS_clusters/output/Coord_cells_DHS_01.csv')
points = dhs_cells[['x', 'y']].drop_duplicates()

# 2. List NetCDF files
nc_files = sorted(glob.glob(r"./ERA5_grid_cell_data/output/010_alltemps_*.nc"))

# 3. Coordinates DataFrame
coords = pd.DataFrame({
    "x": points['x'],
    "y": points['y']
})

# 4. Reference date for column naming
ref_date = pd.Timestamp("1900-01-01")

# 5. Define bins: normal -30–>45 + extremes
bin_definitions = [(T, T+1) for T in range(-30, 45)]
bin_definitions.insert(0, (-float("inf"), -30))   # below -30
bin_definitions.append((45, float("inf")))      # >= 45

# 6. Loop over bins
start = datetime.now()
for i, (low, high) in enumerate(bin_definitions):
    start1 = datetime.now()
    label = f"bin_{low}_{high}"  # unique label for filename
    bin_results = []

    # Loop over NetCDF files
    for f in nc_files:

        # Open dataset
        ds = xr.open_dataset(f)
        ds.close()

        # exctract t2m variable
        da = ds["t2m"]

        # Convert to Celsius
        da = da - 273.15

        # Extract all points (time × points)
        vals = da.sel(
            lon=xr.DataArray(points['x'], dims="points"),
            lat=xr.DataArray(points['y'], dims="points"),
            method="nearest"
        )

        # Extract timestamps
        times = pd.to_datetime(vals["time"].values)

        # Convert to DataFrame (time × points)
        df_temp = pd.DataFrame(vals.values, index=times, columns=points.index)

        # Binarize: 1 if value in [low, high), else 0
        if low == -float("inf"):
            df_bin = (df_temp < high).astype(int)
        elif high == float("inf"):
            df_bin = (df_temp >= low).astype(int)
        else:
            df_bin = ((df_temp >= low) & (df_temp < high)).astype(int)

        # Resample to daily sums (hours per day in this bin)
        df_daily = df_bin.resample("D").sum()

        bin_results.append(df_daily)

        end = datetime.now()

    # Concatenate all months for this bin
    df_bin_all = pd.concat(bin_results)

    # Reformat: rows=points, cols=days
    df_bin_all = df_bin_all.T
    df_bin_all.index.name = "point_id"

    # Convert day index to day numbers since 1900-01-01
    day_numbers = (df_bin_all.columns - ref_date).days+1
    df_bin_all.columns = day_numbers

    # Add coordinates at the front
    out = pd.concat([coords, df_bin_all], axis=1)

    # Save CSV
    out.to_csv("./one_degree_bins_T_WB/output/010_t2m_bins/" + f"t2m_{label}.csv", index = False)
    print(f"Creating {label} took {end - start1}")
    print(f'total time since entering the loop: {end-start}')

Creating bin_-inf_-30 took 0:00:02.400942
total time since entering the loop: 0:00:02.401194
Creating bin_-30_-29 took 0:00:02.363650
total time since entering the loop: 0:00:04.937361
Creating bin_-29_-28 took 0:00:02.294703
total time since entering the loop: 0:00:07.379410
Creating bin_-28_-27 took 0:00:02.285672
total time since entering the loop: 0:00:09.826058
Creating bin_-27_-26 took 0:00:02.301003
total time since entering the loop: 0:00:12.276697
Creating bin_-26_-25 took 0:00:02.342387
total time since entering the loop: 0:00:14.767850
Creating bin_-25_-24 took 0:00:02.416916
total time since entering the loop: 0:00:17.334117
Creating bin_-24_-23 took 0:00:02.332283
total time since entering the loop: 0:00:19.832966
Creating bin_-23_-22 took 0:00:02.338313
total time since entering the loop: 0:00:22.321536
Creating bin_-22_-21 took 0:00:02.341955
total time since entering the loop: 0:00:24.814376
Creating bin_-21_-20 took 0:00:02.372400
total time since entering the loop: 0:

## Then, handle WBGT

In [11]:
# 1. Load the list of DHs cluster points
dhs_cells = pd.read_csv('./matching_grid_cells_DHS_clusters/output/Coord_cells_DHS_0.25.csv')
points = dhs_cells[['x', 'y']].drop_duplicates()

# 2. List NetCDF files
nc_files = sorted(glob.glob(r"./ERA5_grid_cell_data/output/025_alltemps_*.nc"))

# 3. Coordinates DataFrame
coords = pd.DataFrame({
    "x": points['x'],
    "y": points['y']
})

# 4. Reference date for column naming
ref_date = pd.Timestamp("1900-01-01")

# 5. Define bins: normal -30–>45 + extremes
bin_definitions = [(T, T+1) for T in range(-30, 45)]
bin_definitions.insert(0, (-float("inf"), -30))   # below -30
bin_definitions.append((45, float("inf")))      # >= 45

# 6. Loop over bins
start = datetime.now()
for i, (low, high) in enumerate(bin_definitions):
    start1 = datetime.now()
    label = f"bin_{low}_{high}"  # unique label for filename
    bin_results = []

    # Loop over NetCDF files
    for f in nc_files:

        # Open dataset
        ds = xr.open_dataset(f)

        # extract wbgt variable
        da = ds["wbgt"]

        # Convert to Celsius
        da = da - 273.15

        # Extract all points (time × points)
        vals = da.sel(
            lon=xr.DataArray(points['x'], dims="points"),
            lat=xr.DataArray(points['y'], dims="points"),
            method="nearest"
        )

        # Extract timestamps
        times = pd.to_datetime(vals["time"].values)

        # Convert to DataFrame (time × points)
        df_temp = pd.DataFrame(vals.values, index=times, columns=points.index)

        # Binarize: 1 if value in [low, high), else 0
        if low == -float("inf"):
            df_bin = (df_temp < high).astype(int)
        elif high == float("inf"):
            df_bin = (df_temp >= low).astype(int)
        else:
            df_bin = ((df_temp >= low) & (df_temp < high)).astype(int)

        # Resample to daily sums (hours per day in this bin)
        df_daily = df_bin.resample("D").sum()

        bin_results.append(df_daily)

        end = datetime.now()

    # Concatenate all months for this bin
    df_bin_all = pd.concat(bin_results)

    # Reformat: rows=points, cols=days
    df_bin_all = df_bin_all.T
    df_bin_all.index.name = "point_id"

    # Convert day index to day numbers since 1900-01-01
    day_numbers = (df_bin_all.columns - ref_date).days+1
    df_bin_all.columns = day_numbers

    # Add coordinates at the front
    out = pd.concat([coords, df_bin_all], axis=1)

    # Save CSV
    out.to_csv("./one_degree_bins_T_WB/output/025_wbgt_bins/" + f"wbgt_{label}.csv", index = False)
    print(f"Creating {label} took {end - start1}")
    print(f'total time since entering the loop: {end-start}')

Creating bin_-inf_-30 took 0:00:00.209173
total time since entering the loop: 0:00:00.209656
Creating bin_-30_-29 took 0:00:00.179482
total time since entering the loop: 0:00:00.419350
Creating bin_-29_-28 took 0:00:00.181997
total time since entering the loop: 0:00:00.628932
Creating bin_-28_-27 took 0:00:00.186786
total time since entering the loop: 0:00:00.846236
Creating bin_-27_-26 took 0:00:00.177809
total time since entering the loop: 0:00:01.052255
Creating bin_-26_-25 took 0:00:00.173385
total time since entering the loop: 0:00:01.253940
Creating bin_-25_-24 took 0:00:00.170477
total time since entering the loop: 0:00:01.452560
Creating bin_-24_-23 took 0:00:00.159147
total time since entering the loop: 0:00:01.642693
Creating bin_-23_-22 took 0:00:00.156961
total time since entering the loop: 0:00:01.828722
Creating bin_-22_-21 took 0:00:00.156420
total time since entering the loop: 0:00:02.013201
Creating bin_-21_-20 took 0:00:00.164813
total time since entering the loop: 0:

## the following code checks whether all bins sum to 24 for each (point, day) couple

In [5]:
##### TEST FOR TEMPERATURE BINS #####
# 1. List all bin CSV files
csv_files = sorted(glob.glob("./one_degree_bins_T_WB/output/010_t2m_bins/t2m_bin_*.csv"))
print(len(csv_files))
start = datetime.now()

# 2. Load them into DataFrames
dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    dfs.append(df)

# 3. Keep x, y from the first DataFrame
coords = dfs[0][["x", "y"]]

# 4. Drop coords from all DataFrames before summing
dfs_no_coords = [df.drop(columns=["x", "y"]) for df in dfs]

# 5. Sum across bins
df_sum = sum(dfs_no_coords)

# 6. Re-attach coordinates
df_sum = pd.concat([coords, df_sum], axis=1)

# 7. Now df_sum should have 24 everywhere (per day, per point)
df_sum.head()

# 8. Check if everything is 24
check = (df_sum.drop(columns=["x", "y"]) == 24).all().all()
print("All t2m cells sum to 24 ?" , check)
end = datetime.now()
print(f'Test took {end-start}')

77
All t2m cells sum to 24 ? True
Test took 0:00:04.541308


In [12]:
##### TEST FOR WBGT BINS #####
# 1. List all bin CSV files
csv_files = sorted(glob.glob("one_degree_bins_T_WB/output/025_wbgt_bins/wbgt_bin_*.csv"))
print(len(csv_files))
start = datetime.now()

# 2. Load them into DataFrames
dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    dfs.append(df)

# 3. Keep x, y from the first DataFrame
coords = dfs[0][["x", "y"]]

# 4. Drop coords from all DataFrames before summing
dfs_no_coords = [df.drop(columns=["x", "y"]) for df in dfs]

# 5. Sum across bins
df_sum = sum(dfs_no_coords)

# 6. Reattach coordinates
df_sum = pd.concat([coords, df_sum], axis=1)

# 7. Now df_sum should have 24 everywhere (per day, per point)
df_sum.head()

# 8. Check if everything is 24
check = (df_sum.drop(columns=["x", "y"]) == 24).all().all()
print("All wbgt cells sum to 24 ?" , check)
end = datetime.now()
print(f'Test took {end-start}')

77
All wbgt cells sum to 24 ? True
Test took 0:00:01.459364
